# Practical Statistics with Python: Exploring the Penguins Dataset

**Video duration:** 15 minutes  
**Format:** Instructor-led code demonstration  
**Dataset:** Seaborn Penguins dataset

## Introduction

Welcome back to this tutorial!

In this lesson, we'll put statistics into practice using Python and the Penguins dataset.

We'll explore how to summarize data, measure variability, identify potential outliers, investigate relationships between variables, and understand how sample size affects statistical results.

By the end of this practical, you should be able to:

- Load and inspect a dataset using Python.
- Calculate and interpret measures of central tendency and spread.
- Identify potential outliers using the IQR method.
- Explore relationships between numerical variables using correlation and visualisations.
- Simulate repeated sampling and explain how sample size affects the variability of sample means.

Let's get started!

---

## 1:00–2:30 | Import and Inspect the Dataset

Before performing any statistical analysis, we need to understand the dataset we're working with, including its structure, variables, and data quality.

### Code cell 1: Import libraries

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
np.random.seed(42)
```

**Narration:**

First, let's import the libraries we'll need.

We'll use Pandas to manipulate and analyze the dataset, NumPy for numerical operations, and Matplotlib and Seaborn to create visualizations.

### Code cell 2: Load the dataset

```python
df = sns.load_dataset("penguins")

display(df.head())

print("Dataset shape:", df.shape)

print("\nMissing values:\n", df.isnull().sum())
```

**Narration:**

We are using the Penguins dataset available through Seaborn.

Each row represents an individual penguin, and the dataset contains measurements such as bill length, bill depth, flipper length, and body mass, alongside categorical information like species and island.

Let's inspect the first few rows and check the size of our dataset. We also want to check for missing values because incomplete observations can affect our calculations.

We can see that some of the measurements contain missing values. For this practical, we'll remove incomplete rows so that we can work with a complete dataset.

### Code cell 3: Clean the data

```python
df = df.dropna().reset_index(drop=True)

numeric_cols = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

print("Cleaned dataset shape:", df.shape)
```

---

## 2:30–5:00 | Descriptive Statistics

Now that we have a clean dataset, let's begin by understanding the distribution of the measurements.

One of the first things we often want to know when working with numerical data is what a typical observation looks like and how much the observations differ from one another.

We can answer these questions using measures of central tendency and measures of spread.

### Code cell 4: Measures of central tendency

```python
# Mean
print("MEAN")
display(df[numeric_cols].mean().round(2))

# Median
print("MEDIAN")
display(df[numeric_cols].median().round(2))

# Mode
print("MODE")
display(df[numeric_cols].mode().iloc[0].round(2))
```

**Narration:**

Let's calculate the mean, median, and mode for our numerical variables.

The **mean** gives us the average value, the **median** gives us the middle value when the observations are arranged in order, and the **mode** gives us the most frequently occurring value.

These measures help us describe the centre of our data, although they don't always tell us the same thing.

For example, the mean can be influenced by extreme values, while the median is generally more resistant to them.

Comparing the mean and median can therefore help us understand whether our data might be skewed or influenced by unusually large or small observations.

But knowing the centre alone isn't enough. Two datasets can have similar averages while their values are spread out very differently.

Let's look at that next.

### Code cell 5: Measures of spread

```python
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)

spread = pd.DataFrame({
    "Range": df[numeric_cols].max() - df[numeric_cols].min(),
    "Variance": df[numeric_cols].var(),
    "Standard deviation": df[numeric_cols].std(),
    "IQR": q3 - q1
})

display(spread.round(2))
```

**Narration:**

Here, we're calculating several measures of spread.

The **range** tells us the difference between the largest and smallest values. The **variance** measures how dispersed observations are around the mean using squared deviations. The **standard deviation** also measures spread but is expressed in the original units of the variable.

Finally, the **interquartile range**, or IQR, measures the spread of the middle 50 percent of our observations.

Notice that these variables use different units. Body mass is measured in grams, while bill and flipper measurements are in millimetres, so we should always consider the units when interpreting these values.

---

## 5:00–6:30 | Identifying Potential Outliers

We've looked at how values are distributed, but what happens when some observations are unusually far from the rest?

These observations may be potential outliers. They could represent unusual but genuine measurements, or they could indicate errors that need further investigation.

One method we can use to flag potential outliers is the IQR method.

### Code cell 6: Detect outliers using the IQR method

```python
col = "bill_length_mm"

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)

iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = df[
    (df[col] < lower_fence) |
    (df[col] > upper_fence)
]

print("Lower fence:", round(lower_fence, 2))
print("Upper fence:", round(upper_fence, 2))
print("Potential outliers:", len(outliers))

display(outliers[["species", col]])
```

**Narration:**

We are focusing on bill length.

We calculate the first quartile and third quartile, use them to find the IQR, and then calculate the lower and upper boundaries.

Any observation below the lower fence or above the upper fence is flagged as a potential outlier.

However, it's important to remember that an outlier is not automatically an error. It may be a perfectly valid observation, especially when our dataset contains different groups with different characteristics.

We should investigate flagged observations before deciding whether to remove them.

---

## 6:30–9:00 | Exploring Relationships Between Variables

So far, we've been examining individual variables. Let's now shift our focus to understanding how variables relate to one another.

For example, we might want to investigate whether penguins with longer flippers also tend to have greater body mass.

To begin, we can calculate correlations between our numerical variables.

### Code cell 7: Correlation matrix

```python
corr = df[numeric_cols].corr()

plt.figure(figsize=(7, 5))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    fmt=".2f"
)

plt.title("Correlation Between Penguin Measurements")
plt.show()
```

**Narration:**

The correlation coefficient measures the direction and strength of a linear relationship between two numerical variables.

It ranges from negative one to positive one.

A positive correlation means that the variables tend to increase together, while a negative correlation means that as one variable increases, the other tends to decrease.

Values closer to zero indicate a weaker linear relationship.

This heatmap allows us to compare the relationships between several measurements at once.

But rather than relying only on the correlation values, let's visualize one of these relationships to see what it actually looks like.

### Code cell 8: Flipper length vs body mass

```python
r = df["flipper_length_mm"].corr(df["body_mass_g"])

sns.scatterplot(
    data=df,
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species"
)

plt.title(f"Flipper Length vs Body Mass (r = {r:.2f})")
plt.show()
```

**Narration:**

Each point represents an individual penguin. We're comparing its flipper length against its body mass, while the colours distinguish the different species.

Look at the overall pattern. Do penguins with longer flippers generally have greater body mass?

We can observe the relationship, but notice that the species also form distinct groups.

This is important because relationships between variables can sometimes be influenced by other factors. A single correlation coefficient may not fully explain what's happening within different groups of observations.

And remember, correlation does not necessarily imply causation. Observing a relationship between two variables doesn't mean that one variable directly causes changes in the other.

---

## 9:00–12:30 | Simulating Repeated Sampling

Let's now connect what we've been doing to inferential statistics.

In real-world situations, we often don't have access to an entire population. Instead, we collect a sample and use it to estimate or learn something about that population.

But samples can differ. If we select different observations from the same population, we may end up with different sample means.

This is known as **sampling variability**.

Let's demonstrate this using flipper length.

### Code cell 9: Define our population

```python
population = df["flipper_length_mm"]

population_mean = population.mean()

print("Population mean:", round(population_mean, 2))
```

**Narration:**

For this demonstration, we'll treat our cleaned Penguins dataset as the population.

We calculate the overall mean flipper length, which we'll use as a reference when comparing the results from our samples.

### Code cell 10: Draw a single sample

```python
sample = population.sample(n=10, replace=True)

sample_mean = sample.mean()

print("Sample values:\n", sample.to_list())
print("Sample mean:", round(sample_mean, 2))
print("Population mean:", round(population_mean, 2))
```

**Narration:**

Let's randomly select 10 penguins and calculate the mean flipper length for this sample.

Compare our sample mean with the population mean.

They may be close, but they don't have to be exactly equal. This difference occurs because we are using only a subset of observations to represent the entire population.

If we selected another sample of 10 penguins, we might get a different mean.

Let's repeat this process many times to see what happens.

### Code cell 11: Simulate 1,000 sample means

```python
sample_size = 10
n_simulations = 1000

sample_means = []

for _ in range(n_simulations):
    sample = population.sample(n=sample_size, replace=True)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

print("Mean of sample means:", round(sample_means.mean(), 2))
print("Standard deviation of sample means:", round(sample_means.std(), 2))
```

**Narration:**

We're repeating the same process 1,000 times.

Each time, we randomly select 10 observations, calculate their mean, and store that mean.

This gives us 1,000 sample means instead of just one.

The distribution formed by these sample means is called the **sampling distribution of the mean**.

Let's visualize it.

### Code cell 12: Visualize the sampling distribution

```python
sns.histplot(sample_means, bins=25, kde=True)

plt.axvline(
    population_mean,
    linestyle="--",
    label="Population mean"
)

plt.title("Sampling Distribution of the Mean")
plt.xlabel("Sample mean")
plt.legend()
plt.show()
```

**Narration:**

Each bar shows how frequently our simulated sample means fall within a particular range.

The dashed line marks the population mean.

Notice how the sample means tend to cluster around the population mean, even though individual samples produce different results.

This helps us understand that estimates based on samples naturally vary, even when we're sampling from the same population.

---

## 12:30–14:00 | How Sample Size Affects Variability

Now that we've seen how sample means vary across repeated samples, let's investigate another important question.

What happens when we increase the number of observations in each sample?

Would a sample of 5 penguins give us the same consistency as a sample of 50 penguins?

Let's find out.

### Code cell 13: Compare sample sizes

```python
sample_sizes = [5, 10, 50]
n_simulations = 1000

results = {}

for size in sample_sizes:
    means = []

    for _ in range(n_simulations):
        sample = population.sample(n=size, replace=True)
        means.append(sample.mean())

    results[size] = np.array(means)

for size, means in results.items():
    print(
        f"Sample size {size}: "
        f"Mean = {means.mean():.2f}, "
        f"Standard deviation = {means.std():.2f}"
    )
```

**Narration:**

We're repeating the sampling process 1,000 times for each sample size: 5, 10, and 50.

For each sample size, we're calculating the mean of every sample and then measuring the standard deviation of those sample means.

This allows us to compare how much the sample means vary when we use different sample sizes.

### Code cell 14: Visual comparison

```python
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, size in zip(axes, sample_sizes):
    sns.histplot(results[size], bins=25, kde=True, ax=ax)
    ax.axvline(population_mean, linestyle="--")
    ax.set_title(f"Sample size = {size}")
    ax.set_xlabel("Sample mean")

plt.tight_layout()
plt.show()
```

**Narration:**

Let's compare the three distributions.

When we use a sample size of 5, the sample means tend to be more spread out.

As we increase the sample size to 10 and then 50, the sample means generally become more tightly grouped around the population mean.

This demonstrates that larger samples generally provide more consistent estimates of the population mean because they reduce sampling variability.

However, a larger sample alone does not guarantee that our data is representative. How we select our sample also matters.

---

## 14:00–15:00 | Summary and Closing

**Narration:**

Let's recap what we've done in this practical.

We started by loading and inspecting the Penguins dataset, then calculated measures of central tendency and spread to understand the distribution of our numerical variables.

We used the IQR method to identify potential outliers and explored relationships between measurements using correlation and visualizations.

Finally, we simulated repeated sampling to demonstrate sampling variability and observed how increasing sample size generally reduces the variability of sample means.

The key takeaway from this lesson is that statistics is not simply about calculating values. It's about using those values to understand data, identify patterns, evaluate variability, and make informed conclusions while recognizing the uncertainty involved.

That brings us to the end of this practical.

In our next lesson, we'll explore probability distributions and how they help us describe the behaviour of random variables.

---

# Practice Challenge

Using the Penguins dataset, complete the following activities independently.

1. Calculate the mean and median body mass for each penguin species.
2. Identify which numerical variable has the largest standard deviation.
3. Create a box plot of bill length for each penguin species.
4. Calculate the correlation between bill length and bill depth.
5. Create a scatter plot showing the relationship between flipper length and body mass.
6. Repeat the sampling simulation using body mass instead of flipper length.
7. Compare the variability of sample means using sample sizes of 5, 20, and 50.

---

# Reflection Questions

- What did you learn from comparing the mean and median?
- Which variables showed the strongest relationships?
- Why did different samples produce different means?
- What happened to the variability of sample means as sample size increased?
- Why should we consider uncertainty when using sample data to make conclusions about a population?

---

# Instructor Recording Notes

- Keep the notebook prepared with code cells in the order shown.
- Run each cell while explaining its purpose and interpretation.
- Avoid reading every line of code aloud; focus on the key operations.
- Allow students time to interpret outputs and think about what they mean.
- Emphasize interpretation over simply obtaining numerical results.
- The timings are targets. Practise once with a timer and shorten pauses if needed.